# Plotly Word Cloud: Developer Tutorial

This tutorial provides a comprehensive walkthrough of the `plotly_wordcloud.py` module.  
It is designed for developers and contributors who want to understand, use, and potentially extend this component of the Lexos toolkit.

---

## Overview

The `plotly_wordcloud` function generates interactive word clouds using **Plotly**, based on input data that may come from raw text, tokenized lists, `spaCy` documents, term-frequency dictionaries, dataframes, or Lexos’s `DTM` format.

Unlike traditional matplotlib word clouds, this version outputs a **`plotly.graph_objects.Figure`**, allowing for more flexible display, layout manipulation, and interactivity (e.g., hover info, custom HTML saving).

This notebook will walk you through:

- The purpose and structure of `plotly_wordcloud`
- Input types and how they are handled
- Optional arguments for customization
- How the data is converted into Plotly elements
- Limitations and future considerations
- A full example using project text data

---

## Prerequisites

Make sure your environment has the following installed:

- `plotly`
- `pandas`
- `wordcloud`
- `spacy`
- `pydantic`
- Lexos (including `lexos.dtm` and `lexos.visualization.processors`)

> This notebook assumes you are working inside the Lexos project or have it installed locally.

---

## Tutorial Flow

| Section | Topic |
|---------|-------|
| 1 | Introduction and Setup |
| 2 | Understanding Input Types |
| 3 | WordCloud Options and Layout Customization |
| 4 | How Data Is Processed Internally |
| 5 | Plotting and Saving |
| 6 | Example: From Text File to Word Cloud |
| 7 | Notes, Limitations, and Future Ideas |

---


In [ ]:
# Required Imports
from pathlib import Path
from lexos.visualization.plotly_wordcloud import plotly_wordcloud

# You can uncomment and use this to check the environment
import sys
print("Python version:", sys.version)

# Path to the text file
# Update the path below if your .txt file is in a different location
text_path = Path("docs/Austen_Pride.txt")

# Read the content
text = text_path.read_text(encoding="utf-8")

# 🖨️ Preview the first 500 characters
print(text[:500])


Python version: 3.12.3 (main, Apr 15 2024, 17:43:11) [Clang 17.0.6 ]
 Pride and Prejudice
by Jane Austen
Chapter 1
It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife.
However little known the feelings or views of such a man may be on his first entering a neighbourhood, this truth is so well fixed in the minds of the surrounding families, that he is considered the rightful property of some one or other of their daughters.
"My dear Mr. Bennet," said his lady to him one day, "have you heard that Nethe


## Supported Input Types

The `plotly_wordcloud()` function accepts a wide range of data formats, which makes it flexible across NLP pipelines and document processing tasks. Internally, the function detects the input type and processes it accordingly.

### Acceptable formats for `data`:

| Input Type                  | Description |
|----------------------------|-------------|
| `str`                      | A raw text string |
| `list[str]`                | A list of tokenized words |
| `dict[str, int]`           | A dictionary of word frequencies |
| `spacy.tokens.Doc`         | A spaCy document object |
| `spacy.tokens.Span`        | A span from a spaCy document |
| `list[Doc]`, `list[Span]`  | A list of spaCy Docs or Spans |
| `list[list[str]]`          | A list of tokenized documents (as lists of words) |
| `list[Token]`              | A list of spaCy Token objects |
| `list[list[Token]]`        | A list of lists of Tokens |
| `pandas.DataFrame`         | A term-document matrix (terms as index, docs as columns) |
| `lexos.dtm.DTM`            | A Lexos document-term matrix object |

---

### How the function handles each:

- If you pass **text**, it calls `WordCloud.generate_from_text(text)`
- If you pass a **Doc or Span**, it uses a frequency counter over `token.text`
- If you pass **dict, DataFrame, or DTM**, it uses `generate_from_frequencies(...)`
- For **list-based inputs**, helper functions in `lexos.visualization.processors` standardize the format

> 🔍 The input format must be consistent. Mixed-type lists (e.g. `["word", Doc, ["more words"]]`) will raise a `LexosException`.

---

In the upcoming cell, we'll use a `.txt` file as our input and generate the word cloud directly from raw text.


In [4]:
# Generate a basic word cloud using the raw text

fig = plotly_wordcloud(text)

# Note:
# - This will use default WordCloud settings (white background, 2000 max words)
# - It will automatically open a Plotly figure in your notebook or browser (if supported)


## Interpreting the Word Cloud Output

After calling `plotly_wordcloud()` with a valid input, an interactive word cloud is generated using Plotly. Here's how to interpret the visualization:

- **Word Size**: Represents the relative frequency of each word. Larger words appear more often in the input data.
- **Color**: Each word is randomly assigned a color by the `WordCloud` generator. Colors can help visually separate frequent terms.
- **Position**: Words are arranged algorithmically to maximize space usage while avoiding overlap.
- **Hover Info**: Hovering over a word displays its relative frequency (e.g., `"data: 12.50%"`).

This visualization provides an intuitive way to explore prominent terms in your dataset.

> Note: The layout does not include axes or grid lines, as they are intentionally hidden to maintain focus on the words.



---


##  Customizing the Word Cloud: Optional Arguments

The `plotly_wordcloud()` function offers several optional parameters to help you tailor the output to your needs:

---

###  `opts`: WordCloud Options (passed to `WordCloud()`)

Customize the appearance of the word cloud using a dictionary of options.  
These are passed directly to the [`wordcloud.WordCloud`](https://amueller.github.io/word_cloud/generated/wordcloud.WordCloud.html) constructor.

**Example:**
```python
opts = {
    "background_color": "white",
    "max_words": 150,
    "contour_width": 2,
    "contour_color": "navy"
}
````

---

### `layout`: Plotly Layout Customization

You can adjust the layout of the Plotly figure (e.g., width, height, margins, background).
Pass a dictionary of options that updates the default Plotly layout.

**Example:**

```python
layout = {
    "width": 900,
    "height": 600,
    "margin": {"l": 40, "r": 40, "t": 80, "b": 40}
}
```

---

### `path`: Save to File

If you pass a file path (as a string or `Path` object), the function will save the word cloud as an interactive `.html` file.

**Example:**

```python
path = "output/wordcloud.html"
```

---

### `show`: Toggle Display

Controls whether the word cloud is shown immediately or returned as a `go.Figure` object for further use.

* `True` (default): calls `fig.show()`
* `False`: returns the Plotly figure without displaying it

---

### `docs`: Document Selection (for DTM or DataFrame)

Use this to specify one or more documents from a DTM or DataFrame.

* Can be an `int`, `str`, or list of either
* Ignored for other input types

**Example:**

```python
docs = [0, 1]  # Selects the first and second columns of a DTM
```

---

> These optional arguments allow you to fine-tune visual style, selectively generate clouds, or export for later use.



In [7]:
# Customized Word Cloud: Appearance + Save to File

# Define custom WordCloud options
opts = {
    "background_color": "white",
    "max_words": 100,
    "contour_width": 1,
    "contour_color": "darkred"
}

# Define custom Plotly layout
layout = {
    "width": 850,
    "height": 600,
    "margin": {"l": 30, "r": 30, "t": 60, "b": 30},
    "title": {"text": "Custom Word Cloud", "x": 0.5, "xanchor": "center"}
}

# Define output path and ensure the directory exists
path = Path("output/custom_wordcloud.html")
path.parent.mkdir(parents=True, exist_ok=True)

# Generate and save the word cloud
fig = plotly_wordcloud(text, opts=opts, layout=layout, path=path, show=True)



## This cell:

* Uses a max of 100 words
* Adds a red contour to words
* Sets a custom figure size and adds a centered title
* Saves the figure to `output/custom_wordcloud.html`



## How Input Data Is Processed Internally

The `plotly_wordcloud()` function uses a series of internal branches to detect and transform input data into a format compatible with the `WordCloud` library. Here's how it works:

---

### Core Branching Logic

Depending on the input type, the function chooses how to extract or compute word frequencies:

| Input Type       | Processing Method |
|------------------|--------------------|
| `str`            | `WordCloud.generate_from_text()` |
| `Doc` or `Span`  | `Counter([token.text for token in data])` |
| `DTM`            | `processors.process_dtm(data, docs)` |
| `DataFrame`      | `processors.process_dataframe(data, docs)` |
| `list[list[str]]`| `processors.process_list(data, docs)` |
| `list[Doc]` or `list[Span]` | `processors.process_docs(data, docs)` |
| `list[str]`, `list[Token]` | `processors.process_item(data)` |
| `dict[str, int]` | Used directly as word frequencies |

---

### What the Processors Do

Each `processors.*` function standardizes its input and returns a dictionary in the format:

```python
{"word1": count1, "word2": count2, ...}

```

This format is compatible with:

```python
WordCloud(**opts).generate_from_frequencies(counts)
```

If the input type is unsupported or mixed (e.g., a list of strings and Docs together), a LexosException is raised to prevent misinterpretation.


In [8]:
# Peek Inside: Extracted WordCloud Layout Data

from wordcloud import WordCloud
from collections import Counter

# Step 1: Generate the WordCloud object from text
wc = WordCloud(background_color="white", max_words=50).generate(text)

# Step 2: Manually inspect the layout_ attribute
layout_data = wc.layout_

# Step 3: Display first few layout entries
for i, ((word, freq), fontsize, position, orientation, color) in enumerate(layout_data[:5]):
    print(f"{i+1}. Word: {word}")
    print(f"   Frequency: {freq:.4f}")
    print(f"   Font size: {fontsize}")
    print(f"   Position: {position}")
    print(f"   Orientation: {orientation}")
    print(f"   Color: {color}")
    print("---")


1. Word: Elizabeth
   Frequency: 1.0000
   Font size: 68
   Position: (np.int64(12), np.int64(11))
   Orientation: None
   Color: rgb(244, 230, 30)
---
2. Word: will
   Frequency: 0.7552
   Font size: 60
   Position: (np.int64(138), np.int64(15))
   Orientation: None
   Color: rgb(92, 200, 99)
---
3. Word: much
   Frequency: 0.5996
   Font size: 54
   Position: (np.int64(107), np.int64(149))
   Orientation: None
   Color: rgb(58, 84, 140)
---
4. Word: said
   Frequency: 0.5844
   Font size: 53
   Position: (np.int64(69), np.int64(349))
   Orientation: 2
   Color: rgb(53, 94, 141)
---
5. Word: must
   Frequency: 0.5750
   Font size: 53
   Position: (np.int64(83), np.int64(17))
   Orientation: None
   Color: rgb(110, 206, 88)
---


## This gives insight into:

The word list used

Their frequencies and sizes

Where they are placed (position)

Their color

Orientation (often None — a limitation of WordCloud.layout_)

## Constructing the Plotly Figure

Once the layout information is extracted from the `WordCloud` object, the function builds a Plotly `go.Figure` using a `Scatter` trace with mode `"text"`.

---

### Data Extracted from `wc.layout_`

Each entry in `wc.layout_` provides:

- `word`: The actual text
- `freq`: The normalized frequency (0–1)
- `fontsize`: Calculated font size based on frequency
- `position`: (x, y) location on canvas
- `orientation`: Typically `None` (due to a limitation in `WordCloud`)
- `color`: Assigned font color

These values are collected into separate lists to form the trace:

```python
go.Scatter(
    x=x_positions,
    y=y_positions,
    text=word_list,
    textfont=dict(size=fontsize_list, color=color_list),
    hoverinfo="text",
    hovertext=[f"{word}: {freq:.2%}" for word, freq in zip(word_list, freq_list)],
    mode="text"
)


## Layout Settings
The layout is also customized with:

No visible axes (showgrid: False, showticklabels: False)

Fixed width and height (defaults to 750x750)



## Output Format
The final output is a go.Figure object, which:

Can be displayed with .show()

Can be saved to .html using .write_html(path)

Can be further modified using any Plotly method

##  Known Limitations and Future Considerations

While `plotly_wordcloud()` offers a flexible, interactive alternative to traditional word clouds, it also comes with some constraints due to underlying library limitations:

---

###  Orientation Limitation

The `WordCloud.layout_` attribute does **not retain orientation data**, so all words in the Plotly visualization are displayed horizontally.

- `orientation` values are present in the layout tuple but are typically `None`
- Vertical or angled word support is not currently feasible with Plotly

---

###  Layout Precision

Word positioning is determined by the internal `WordCloud` layout logic and translated approximately to Plotly coordinates.  
This means:
- Overlap prevention may not be perfect
- Word spacing is not pixel-perfect

---

###  Class Refactor Suggestion

The current implementation is a **standalone function**. However, refactoring it as a class could provide:

- Better encapsulation of configuration and state
- Cleaner separation between preprocessing, WordCloud generation, and Plotly rendering
- Easier extension (e.g., adding export methods like `.to_image()`)

---

###  Limited Interactivity

While Plotly supports interactive features like hover info and zooming, it does **not support dynamic reshuffling** of word positions or real-time filtering.

---

Despite these limitations, this tool remains highly effective for exploring textual patterns in a visually appealing and customizable way.
